# Tuning the combined reading

The first notebook settled which approach wins: **the room-finder knows which rooms exist,
our wall model knows where the walls are, and the plan's own printed names beat any guess
about what a room is called.** That combination is now what the pipeline runs.

This notebook tunes it. Four settings, changed one at a time, each measured on the same 25
plans and shown side by side against the default.

**No GPU needed.** The room-finder already ran — this notebook reads its answers out of the
zip that notebook produced and only re-does the cheap part.

**Before you start:** have `results.zip` from `plan_reading_modal.ipynb` to hand. About ten
minutes.

---

### What is being tuned

Growing a room out to the walls has two settings that matter, and neither had ever been
measured.

**How far in to pull the starting point.** The room-finder's outline is coarse, so its edge
routinely lands *on* a wall or slightly past it. Starting a room from a shape that touches a
wall lets it grow straight through into the room next door, so we shrink each one before
growing it. Shrink too little and rooms leak into each other; too much and small rooms
vanish altogether.

**Where a room is allowed to go.** Today a room may grow anywhere inside the *convex hull*
of the drawing — and the hull of an L-shaped or bay-fronted plan includes the garden in the
crook of the L. Measured across these plans it is on average **seven times** the real
footprint, which is why rooms sometimes shoot out into the margin as long thin spikes.
There is now an alternative that follows the building's real outline; it is off by default
because it does not work on every plan, and this notebook is how we find out whether it
earns being on.

## 1 · Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/root/tuning")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)
OURS = ROOT / "visit-it"

if not OURS.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/romainbigare/visit-it.git", str(OURS)],
                   capture_output=True, text=True)
print("visit-it:", "ready" if OURS.exists() else "FAILED to clone")

import numpy
PIN = f"numpy=={numpy.__version__}"


def pip_install(*packages):
    """Install with NumPy held still — moving it breaks every binary package."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", PIN, *packages],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-1200:], r.stderr[-1200:])
        raise SystemExit(f"could not install {packages}")


pip_install("opencv-python-headless", "scikit-image", "scipy", "shapely",
            "segmentation-models-pytorch", "safetensors", "pytesseract", "timm")
if subprocess.run(["which", "tesseract"], capture_output=True).returncode:
    subprocess.run(["apt-get", "-qq", "update"], capture_output=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "tesseract-ocr"], capture_output=True)
print("setup done")

### The plans and the wall model

In [ ]:
import json
import ssl
import urllib.request
import warnings

sys.path.insert(0, str(OURS))
warnings.filterwarnings("ignore")

from pipeline.floorplan import ocr as ocr_mod
from pipeline.floorplan import preprocess, vectorise, wallnet

PLANS = ROOT / "plans"
PLANS.mkdir(exist_ok=True)
golden = json.loads((OURS / "data" / "golden" / "golden_set.json").read_text())
ctx = ssl.create_default_context()
for listing in golden["listings"]:
    plans = listing.get("floorplans") or []
    dest = PLANS / f"{listing['listing_id']}.png"
    if not plans or dest.exists():
        continue
    try:
        req = urllib.request.Request(plans[0]["url"], headers={"User-Agent": "Mozilla/5.0"})
        dest.write_bytes(urllib.request.urlopen(req, context=ctx, timeout=60).read())
    except Exception as exc:
        print("  could not fetch", listing["listing_id"], exc)

(OURS / "models").mkdir(exist_ok=True)
wallnet.MODEL_PATH = OURS / "models" / "plan_walls.safetensors"
if not wallnet.MODEL_PATH.exists():
    urllib.request.urlretrieve(wallnet.MODEL_URL, wallnet.MODEL_PATH)
assert wallnet.available(), "the wall model did not load"

IDS = sorted(p.stem for p in PLANS.glob("*.png"))
print(f"{len(IDS)} plans, wall model ready")

### The room-finder's answers

Upload the `results.zip` from `plan_reading_modal.ipynb`, then pick the reading its table
said was best at **finding rooms** — that is all these are used for here, since their
corners get replaced either way.

In [ ]:
import shutil
import zipfile

# Upload results.zip through the sidebar file browser, then point this at it.
ARCHIVE = ROOT / "results.zip"
if not ARCHIVE.exists():
    found = sorted(Path("/root").rglob("results.zip")) + sorted(Path("/mnt").rglob("results.zip"))
    if found:
        shutil.copy(found[0], ARCHIVE)
        print(f"found {found[0]}")
assert ARCHIVE.exists(), (
    "upload results.zip using the file browser in the sidebar, then re-run this cell")

with zipfile.ZipFile(ARCHIVE) as z:
    readings = {Path(n).stem: n for n in z.namelist()
                if n.endswith(".json") and Path(n).stem != "ladder"}
print("readings in the zip:")
for name in sorted(readings):
    print("  ", name)

# The one the model produced on its own. Change if the table named a different one.
PICK = next((n for n in ("asked_three_ways_kept_the_agreement",
                         "second_opinion_cleaned_up_picture",
                         "second_opinion_plans_as_published")
             if n in readings), None)
assert PICK, f"none of the expected readings are in the zip — pick one from {sorted(readings)}"
with zipfile.ZipFile(ARCHIVE) as z:
    PREDICTIONS = json.loads(z.read(readings[PICK]))
print(f"\nusing: {PICK}  ({sum(len(v['rooms']) for v in PREDICTIONS.values())} rooms "
      f"across {len(PREDICTIONS)} plans)")

### Everything each plan needs, worked out once

Straightening the plan, reading its text and running the wall model take about ten seconds
a plan and give the same answer every time. Doing them once means each setting below takes
seconds rather than minutes.

In [ ]:
import time

import numpy as np
from PIL import Image

PREP = {}
t0 = time.time()
for lid in IDS:
    if lid not in PREDICTIONS:
        continue
    pi = preprocess.prepare(PLANS / f"{lid}.png")
    text = ocr_mod.read(pi.rgb)
    walls = wallnet.barrier(pi.rgb, pi.ink, text.words, pi.wall_half_px)
    if walls is None or not walls.any():
        continue
    rooms = [r for r in PREDICTIONS[lid]["rooms"] if len(r.get("polygon_px") or []) >= 3]
    if not rooms:
        continue
    PREP[lid] = {
        "pi": pi, "text": text, "walls": walls,
        "seeds": vectorise.roomfinder.to_geometry([r["polygon_px"] for r in rooms], pi),
        "names": [str(r.get("label") or "").lower() for r in rooms],
    }
print(f"prepared {len(PREP)} plans in {time.time() - t0:.0f}s")

### The score

In [ ]:
LADDER = []


def wall_match(reading):
    per_plan = {}
    for lid, rooms in reading.items():
        ref = PREP[lid]["walls"]
        polys = [r["polygon_px"] for r in rooms if len(r["polygon_px"]) >= 3]
        s = wallnet.outline_on_wall(polys, ref)
        if s:
            per_plan[lid] = float(np.median(s))
    return per_plan


def run(name, note, **kwargs):
    """One setting, across every plan, scored and added to the table."""
    reading = {}
    t0 = time.time()
    for lid, prep in PREP.items():
        try:
            v = vectorise.segment_from_room_seeds(
                prep["pi"], prep["text"], prep["seeds"], mask=prep["walls"],
                fallback_labels=prep["names"], **kwargs)
        except Exception as exc:
            print(f"  {lid}: {type(exc).__name__}: {exc}")
            continue
        if v is None or not v.rooms:
            continue
        reading[lid] = [{"polygon_px": r.polygon_px, "label": r.label or "",
                         "seeded_by": r.seeded_by} for r in v.rooms]

    per_plan = wall_match(reading)
    if not per_plan:
        print(f"{name}: produced nothing")
        return None
    rooms = sum(len(v) for v in reading.values())
    printed = sum(1 for v in reading.values() for r in v if r["seeded_by"] == "caption")
    entry = {"name": name, "note": note, "kwargs": kwargs, "reading": reading,
             "score": float(np.median(list(per_plan.values()))), "per_plan": per_plan,
             "rooms": rooms, "printed": printed}
    LADDER.append(entry)
    base = LADDER[0]["score"] if len(LADDER) > 1 else None
    print(f"  wall match {entry['score']:.0%}   ·  {rooms} rooms  ·  "
          f"{printed} named from the plan   ({time.time() - t0:.0f}s)")
    if base is not None:
        d = entry["score"] - base
        print(f"  default is {base:.0%}  →  "
              f"{'better' if d > 0.005 else 'worse' if d < -0.005 else 'no change'}"
              f" ({d:+.0%})")
    return entry

---

## 2 · The settings, one at a time

Every one of these runs on all 25 plans and prints its score against the default.

### Default — what the pipeline runs today

Starting points shrunk to 45% of each room's width, rooms free to grow anywhere inside the
convex hull of the drawing.

In [ ]:
run("Default (what ships today)",
    "starting points at 45%, rooms free inside the drawing's convex hull")

### A · Shrink the starting points less

At 25% the starting point stays closer to the room the model actually found, which should
help small rooms survive — and risks a room touching a wall and leaking through it.

In [ ]:
run("A · starting points at 25%",
    "closer to the model's own outline — better for small rooms, riskier for leaks",
    seed_core=0.25)

### B · Shrink them more

At 65% the starting point is a small core in the middle of each room. Safe against leaks; small rooms may not survive it at all.

In [ ]:
run("B · starting points at 65%",
    "a small core in the middle of each room — safe against leaks, hard on small rooms",
    seed_core=0.65)

### C · Keep rooms inside the building

Instead of the convex hull, follow the walls' real outline — so a bay window or the crook
of an L is outside the building and a room cannot grow into it.

This does not work on every plan: sealing a thin-line drawing shut can fail, and flooding in
from the page edge leaks through any external door. Where both look wrong it falls back to
today's behaviour, so the worst case is no change rather than lost rooms.

In [ ]:
run("C · rooms confined to the building",
    "the walls' real outline instead of the convex hull",
    confine=True)

### D · Both of the ones that helped

Whichever starting-point size won, together with confining rooms to the building.

In [ ]:
sizes = [e for e in LADDER if "starting points" in e["name"]]
best_size = max(sizes, key=lambda e: e["score"]) if sizes else None
core = best_size["kwargs"].get("seed_core") if best_size else None
label = f"{core:.0%}" if core else "45%"
print(f"best starting-point size so far: {label}")
run(f"D · starting points at {label}, confined to the building",
    "the two changes above, together",
    **({"seed_core": core} if core else {}), confine=True)

---

## 3 · The report

### The table

In [ ]:
best = max(LADDER, key=lambda e: e["score"])
default = LADDER[0]

print(f'{"":<3}{"setting":<46}{"wall match":>11}{"vs default":>12}{"rooms":>7}{"named":>7}')
print("-" * 86)
for i, e in enumerate(LADDER):
    delta = "—" if i == 0 else f'{e["score"] - default["score"]:+.0%}'
    mark = " ←" if e is best else ""
    print(f'{i:<3}{e["name"][:45]:<46}{e["score"]:>10.0%}{delta:>12}'
          f'{e["rooms"]:>7}{e["printed"]:>7}{mark}')
print("-" * 86)
print(f'\nbest: {best["name"]} — {best["score"]:.0%}')
print(f'({best["note"]})')

# A higher score with fewer rooms is not obviously a win. The score asks whether a
# room's edge sits on a wall; deleting an awkward room raises it just as surely as
# fixing one does, and only the pictures can tell those apart.
lost = default["rooms"] - best["rooms"]
if lost > 0.1 * default["rooms"]:
    print(f'\n  CAREFUL: it also found {lost} fewer rooms than the default '
          f'({best["rooms"]} against {default["rooms"]}).')
    print('  A setting that deletes an awkward room scores better for doing so.')
    print('  Look at the pictures below before taking this one.')
elif lost > 0:
    print(f'  ({lost} fewer rooms than the default — worth a glance, not a worry)')

if best is default:
    print("\nNothing beat the default. That is a result: the settings are already right,")
    print("and the next improvement has to come from somewhere else.")
else:
    args = ", ".join(f"{k}={v!r}" for k, v in best["kwargs"].items())
    print(f'\nTo make it the default, set these in pipeline/floorplan/vectorise.py:')
    for k, v in best["kwargs"].items():
        const = "SEED_CORE_FRACTION" if k == "seed_core" else "CONFINE_TO_FOOTPRINT"
        print(f'    {const} = {v!r}')

### The chart

In [ ]:
import matplotlib.pyplot as plt

# One measure, one series, so one colour for every bar — the length is the message.
BAR = "#2a78d6"
INK, MUTED, RULE = "#0b0b0b", "#52514e", "#d8d8d2"

names = [e["name"] for e in LADDER]
scores = [e["score"] for e in LADDER]
y = np.arange(len(LADDER))

fig, ax = plt.subplots(figsize=(10, 0.62 * len(LADDER) + 2.1))
fig.patch.set_facecolor("#fcfcfb")
ax.set_facecolor("#fcfcfb")
ax.barh(y, scores, height=0.6, color=BAR, zorder=3)
ax.axvline(scores[0], color=MUTED, linestyle="--", linewidth=1, zorder=2)
ax.text(scores[0], -0.75, "default", color=MUTED, fontsize=9, ha="center", va="bottom")

for i, s in enumerate(scores):
    ax.text(s + 0.012, i, f"{s:.0%}" + ("   best" if LADDER[i] is best else ""),
            va="center", fontsize=10, color=INK,
            fontweight="bold" if LADDER[i] is best else "normal")

ax.set_yticks(y); ax.set_yticklabels(names, fontsize=10, color=INK)
ax.invert_yaxis()
ax.set_xlim(0, max(scores) * 1.22)
ax.set_xticks(np.arange(0, 1.01, 0.2))
ax.set_xticklabels([f"{v:.0%}" for v in np.arange(0, 1.01, 0.2)], color=MUTED, fontsize=9)
ax.set_xlabel("share of every room's edge that lands on a wall", color=MUTED, fontsize=10)
ax.grid(axis="x", color=RULE, linewidth=1, zorder=0)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color(RULE)
ax.tick_params(length=0)
ax.set_title(f"Each setting, on the same {len(PREP)} plans",
             fontsize=13, color=INK, loc="left", pad=22)
plt.tight_layout(); plt.show()

### Every plan: the default beside the best

Left the plan, middle what ships today, right the best setting above. If they look the same,
the change did nothing worth having — which the table will already have said, but it is
worth seeing.

What to look for:

- rooms that **shoot out past the building** into the margin — that is what setting C exists
  to stop, and the score cannot see it
- **small rooms** — a WC or a cupboard present in one panel and gone in the other
- rooms that have **leaked into their neighbour** through a doorway

In [ ]:
import colorsys

from matplotlib.patches import Polygon as MplPoly


def hue(i):
    return colorsys.hsv_to_rgb((i * 0.61803) % 1.0, 0.55, 0.95)


def draw(ax, img, rooms, title):
    ax.imshow(img); ax.axis("off"); ax.set_title(title, fontsize=10)
    for i, room in enumerate(rooms):
        p = np.asarray(room["polygon_px"])
        if len(p) < 3:
            continue
        c = hue(i)
        ax.add_patch(MplPoly(p, closed=True, facecolor=c + (0.35,), edgecolor=c, linewidth=2))
        if room.get("label"):
            ax.text(*p.mean(axis=0), room["label"], ha="center", va="center", fontsize=7.5,
                    weight="bold",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", ec="none", alpha=0.75))


for lid in sorted(PREP):
    original = np.array(Image.open(PLANS / f"{lid}.png").convert("RGB"))
    geom = PREP[lid]["pi"].rgb
    d_rooms = default["reading"].get(lid, [])
    b_rooms = best["reading"].get(lid, [])
    fig, axes = plt.subplots(1, 3, figsize=(19, 6.2))
    axes[0].imshow(original); axes[0].axis("off")
    axes[0].set_title(f"{lid} — the plan", fontsize=10)
    draw(axes[1], geom, d_rooms,
         f'default · {len(d_rooms)} rooms · {default["per_plan"].get(lid, 0):.0%}')
    draw(axes[2], geom, b_rooms,
         f'{best["name"][:34]} · {len(b_rooms)} rooms · {best["per_plan"].get(lid, 0):.0%}')
    plt.tight_layout(); plt.show()

### Take it home

In [ ]:
import shutil

OUT = ROOT / "tuning_results"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir()
(OUT / "settings.json").write_text(json.dumps(
    [{k: v for k, v in e.items() if k not in ("reading",)} for e in LADDER], indent=1))
for e in LADDER:
    slug = e["name"].lower().replace(" ", "_").replace("·", "").replace("%", "pct")
    slug = "".join(ch for ch in slug if ch.isalnum() or ch == "_").strip("_")
    (OUT / f"{slug}.json").write_text(json.dumps(e["reading"], indent=1))

archive = shutil.make_archive(str(ROOT / "tuning_results"), "zip", OUT)
print(f"written to  {archive}")
print("Download it from the file browser in the sidebar.")

---

## What to do with this

**If a setting beat the default**, the table prints the two lines to change in
`pipeline/floorplan/vectorise.py`. Change them, re-run the pipeline, and check the
before-and-after in `python -m tools.plan_vs_shell build`.

**If nothing beat it**, that is worth knowing rather than a disappointment: the settings are
already right, and the next improvement has to come from somewhere else — a better
room-finder, or teaching the wall model our own plans
(`notebooks/finetune_wallnet_colab.ipynb`).

**Either way, look at the pictures.** The score asks whether a room's edge sits on a wall,
and a room that has ballooned out past the building has its edge on the *outside* of one, so
it scores well while being plainly wrong. That is the one failure this measurement cannot
see, and setting C is the attempt to stop it happening at all.